In [2]:
# OUR CUSTOM ALGORITHM (MCRI) OVERVIEW
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import BallTree
from sklearn.metrics import roc_auc_score, accuracy_score

print("Applying our Custom MCRI Algorithm...")
csv_path = 'pfas_major_compounds.csv'
df = pd.read_csv(csv_path)

df_pivot = df.pivot_table(index=['lat', 'lon', 'type'], columns='substance', values='value', aggfunc='mean').fillna(0).reset_index()

compound_cols = df_pivot.columns[3:]  
df_pivot['total_pfas'] = df_pivot[compound_cols].sum(axis=1)
df_pivot['compound_count'] = (df_pivot[compound_cols] > 0).sum(axis=1)

weights = {'PFOS': 0.4, 'PFOA': 0.35, 'PFHxS': 0.25}
df_pivot['toxicity_score'] = sum(df_pivot.get(comp, 0) * w for comp, w in weights.items())

if 'PFOS' in df_pivot.columns and 'PFOA' in df_pivot.columns:
    df_pivot['PFOS_PFOA_interaction'] = df_pivot['PFOS'] * df_pivot['PFOA']
else:
    df_pivot['PFOS_PFOA_interaction'] = 0

df_pivot['MCRI'] = (0.3 * df_pivot['total_pfas'] + 0.3 * df_pivot['toxicity_score'] + 0.2 * df_pivot['compound_count'] + 0.2 * df_pivot['PFOS_PFOA_interaction'])

le = LabelEncoder()
df_pivot['media_encoded'] = le.fit_transform(df_pivot['type'])

# Custom dynamic hotspot distance
hotspot_threshold = df_pivot['MCRI'].quantile(0.95)
hotspots_df = df_pivot[df_pivot['MCRI'] >= hotspot_threshold][['lat', 'lon']].copy()
tree = BallTree(np.deg2rad(hotspots_df[['lat', 'lon']].values), metric='haversine')
query_points = np.deg2rad(df_pivot[['lat', 'lon']].values)
dist, _ = tree.query(query_points, k=1)
df_pivot['dist_to_nearest_hotspot_km'] = dist.flatten() * 6371.0

# Define Target using our strategy
mcri_threshold = df_pivot['MCRI'].quantile(0.75)
df_pivot['high_risk_label'] = (df_pivot['MCRI'] >= mcri_threshold).astype(int)

feature_columns = ['lat', 'lon', 'media_encoded', 'dist_to_nearest_hotspot_km']
X = df_pivot[feature_columns]
y = df_pivot['high_risk_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs('our_model', exist_ok=True)

print("\n--- TRAINING ALL 4 MODELS ON OUR CUSTOM ALGORITHM ---")
# Extra Trees
print("1. Extra Trees")
mcri_et = ExtraTreesClassifier(n_estimators=300, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
mcri_et.fit(X_train, y_train)
joblib.dump(mcri_et, 'our_model/our_extratrees.pkl')

# KNN
print("2. KNN")
mcri_knn = KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
mcri_knn.fit(X_train_scaled, y_train)
joblib.dump(mcri_knn, 'our_model/our_knn.pkl')

# Random Forest
print("3. Random Forest")
mcri_rf = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
mcri_rf.fit(X_train, y_train)
joblib.dump(mcri_rf, 'our_model/our_randomforest.pkl')

# XGBoost
print("4. XGBoost")
pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
mcri_xgb = XGBClassifier(scale_pos_weight=pos_weight, n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss')
mcri_xgb.fit(X_train, y_train)
joblib.dump(mcri_xgb, 'our_model/our_xgboost.pkl')

print("\nALL 4 MODELS SAVED TO 'our_model/' folder.")

print("\n--- MODEL PERFORMANCE METRICS ---")
results = []
results.append(('Extra Trees', roc_auc_score(y_test, mcri_et.predict_proba(X_test)[:, 1]), accuracy_score(y_test, mcri_et.predict(X_test))))
results.append(('KNN', roc_auc_score(y_test, mcri_knn.predict_proba(X_test_scaled)[:, 1]), accuracy_score(y_test, mcri_knn.predict(X_test_scaled))))
results.append(('Random Forest', roc_auc_score(y_test, mcri_rf.predict_proba(X_test)[:, 1]), accuracy_score(y_test, mcri_rf.predict(X_test))))
results.append(('XGBoost', roc_auc_score(y_test, mcri_xgb.predict_proba(X_test)[:, 1]), accuracy_score(y_test, mcri_xgb.predict(X_test))))

results.sort(key=lambda x: x[1], reverse=True)
for rank, (name, auc, acc) in enumerate(results):
    print(f"#{rank+1} -> {name:20s} | ROC-AUC: {auc:.4f} | Accuracy: {acc:.4f}")

champion_name = results[0][0]
print(f"\n🏆 CHAMPION IDENTIFIED: {champion_name} 🏆")

Applying our Custom MCRI Algorithm...

--- TRAINING ALL 4 MODELS ON OUR CUSTOM ALGORITHM ---
1. Extra Trees
2. KNN
3. Random Forest
4. XGBoost

ALL 4 MODELS SAVED TO 'our_model/' folder.

--- MODEL PERFORMANCE METRICS ---
#1 -> Random Forest        | ROC-AUC: 0.8965 | Accuracy: 0.8531
#2 -> XGBoost              | ROC-AUC: 0.8848 | Accuracy: 0.8157
#3 -> KNN                  | ROC-AUC: 0.8735 | Accuracy: 0.8427
#4 -> Extra Trees          | ROC-AUC: 0.8554 | Accuracy: 0.7223

🏆 CHAMPION IDENTIFIED: Random Forest 🏆
